# BiRRT vs. OMPL motion planning

Both `BiRRTPlanner` and `OMPLPlanner` satisfy the same `MotionPlanner` interface and plan over the same `ConfigurationSpace`, so we can swap one for the other. This notebook plans the Panda arm around an obstacle with each, compares planning time, and renders the resulting plans side by side.

Run top to bottom (`make_panda` compiles IKFast on first use).

In [ ]:
import time

import numpy as np
import pybullet as p
from spatialmath import SE3

from prpl_kinematics.collision import PyBulletCollisionChecker
from prpl_kinematics.geometry.shapes import BoxShape
from prpl_kinematics.planning import BiRRTPlanner, OMPLPlanner
from prpl_kinematics.robots import make_panda
from prpl_kinematics.tree.joints import FixedJoint
from prpl_kinematics.tree.kinematic_tree import Edge, Node

robot = make_panda()
arm = robot.groups["arm"]
start = robot.home
goal = {**dict(start), "panda_joint1": [1.4]}

# Place a box where the gripper passes at mid-swing, so the straight joint-space
# path is blocked and both planners must detour around it.
mid_swing = {**dict(start), "panda_joint1": [0.7]}
obstacle_at = robot.tree.forward_kinematics(robot.manipulators["arm"].ee_frame, mid_swing).t
block = BoxShape(size=(0.12, 0.12, 0.5))
robot.tree.add_node(Node("obstacle", visuals=[block], collisions=[block]))
robot.tree.add_edge(
    Edge(robot.tree.root, "obstacle", FixedJoint(name="ofix", origin=SE3(*obstacle_at)))
)

checker = PyBulletCollisionChecker(p.connect(p.DIRECT))
checker.load(robot.tree)
checker.ignore(robot.allowed_collision_pairs)
assert not checker.in_collision(start) and not checker.in_collision(goal)

## Planning time

Plan with each planner several times (varying the random seed) and compare the mean wall-clock time and path length. `BiRRTPlanner` wraps `prpl_utils.BiRRT`; `OMPLPlanner` wraps OMPL's `RRTConnect` and simplifies the result.

In [ ]:
def benchmark(make_planner, trials=5):
    times, lengths, paths = [], [], []
    for seed in range(trials):
        planner = make_planner(np.random.default_rng(seed))
        t0 = time.perf_counter()
        path = planner.plan(start, goal)
        times.append(time.perf_counter() - t0)
        assert path is not None and all(not checker.in_collision(c) for c in path)
        lengths.append(len(path))
        paths.append(path)
    return times, lengths, paths


birrt_times, birrt_lengths, birrt_paths = benchmark(
    lambda rng: BiRRTPlanner(arm, checker.in_collision, rng, num_iters=1000)
)
ompl_times, ompl_lengths, ompl_paths = benchmark(
    lambda rng: OMPLPlanner(arm, checker.in_collision, rng, timeout=5.0)
)

print(f"{'planner':<8} {'mean time (ms)':>16} {'mean path length':>18}")
print(f"{'BiRRT':<8} {1000 * np.mean(birrt_times):>16.1f} {np.mean(birrt_lengths):>18.1f}")
print(f"{'OMPL':<8} {1000 * np.mean(ompl_times):>16.1f} {np.mean(ompl_lengths):>18.1f}")

## Render the plans

Render each plan as an inline animation. The renderer uses its own PyBullet client (separate from the collision checker, so the checker's bodies are not drawn over the robot).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

from prpl_kinematics.visualization import CameraParams, PyBulletRenderer, render_configurations

renderer = PyBulletRenderer(p.connect(p.DIRECT))
renderer.load(robot.tree)
camera = CameraParams(target=(0.25, 0.1, 0.45), distance=1.5, yaw=60.0, pitch=-40.0)


def plan_video(label, path):
    images = render_configurations(renderer, path, camera)
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.set_title(label)
    ax.axis("off")
    canvas = ax.imshow(images[0])

    def update(i):
        canvas.set_data(images[i])
        return [canvas]

    anim = animation.FuncAnimation(fig, update, frames=len(images), interval=60, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


display(plan_video("BiRRT", birrt_paths[0]))
display(plan_video("OMPL", ompl_paths[0]))